# 03 — Holdout Test, Uplift & Campaign ROI

Goal: simulate a randomized test/control campaign, quantify incremental conversion lift, estimate uncertainty and calculate incremental margin versus campaign spend.

## 1. Simulate randomized test and control groups
Assume campaign target is a reactivation segment. Test group receives a message; control does not. The campaign date and attribution window are treated as fixed in the simulation.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
n_test,n_control=50000,6000
p_control,p_test=0.021,0.025
control_conv=np.random.binomial(n_control,p_control)
test_conv=np.random.binomial(n_test,p_test)
results=pd.DataFrame({'group':['Test (message)','Control (holdout)'],'n':[n_test,n_control],'conversions':[test_conv,control_conv]})
results['conversion']=results.conversions/results.n
results

## 2. Uplift, incremental orders and 95% confidence interval

In [ ]:
pt,pc=results.loc[0,'conversion'],results.loc[1,'conversion']
uplift=pt-pc
se=np.sqrt(pt*(1-pt)/n_test+pc*(1-pc)/n_control)
ci_low,ci_high=uplift-1.96*se,uplift+1.96*se
incremental_orders=uplift*n_test
print(f'Test conversion: {pt:.2%}; control: {pc:.2%}')
print(f'Uplift: {uplift*100:.2f} percentage points')
print(f'95% CI: [{ci_low*100:.2f}, {ci_high*100:.2f}] pp')
print(f'Estimated incremental orders: {incremental_orders:,.0f}')

## 3. Statistical significance (two-proportion z-test)
A p-value below 0.05 is a conventional threshold for evidence against equal conversion rates; do not treat it as proof of business impact.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
z,p_value=proportions_ztest([test_conv,control_conv],[n_test,n_control])
print(f'z statistic = {z:.3f}; p-value = {p_value:.4f}')
print('Significant at 5%:',p_value<0.05)

## 4. Incremental margin, costs and ROI
Illustrative assumptions: margin per incremental order = 450 ₽; message cost = 2.2 ₽ per recipient. ROI here is incremental margin / campaign cost (not net ROI).

In [ ]:
margin_per_order=450
message_cost_per_recipient=2.2
campaign_cost=n_test*message_cost_per_recipient
incremental_margin=incremental_orders*margin_per_order
roi_ratio=incremental_margin/campaign_cost if campaign_cost else np.nan
net_profit=incremental_margin-campaign_cost
print(f'Campaign cost: {campaign_cost:,.0f} ₽')
print(f'Incremental margin: {incremental_margin:,.0f} ₽')
print(f'Net incremental profit: {net_profit:,.0f} ₽')
print(f'Margin / cost: {roi_ratio:.2f}x')

## 5. Visualize group conversion and uplift interval

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(results.group,results.conversion*100,yerr=[0,0],capsize=4)
ax.set_ylabel('Conversion (%)'); ax.set_title('Test vs control conversion')
for i,v in enumerate(results.conversion):ax.text(i,v*100+0.08,f'{v:.2%}',ha='center')
plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(7,2.8))
ax.errorbar([0],[uplift*100],yerr=[[ (uplift-ci_low)*100 ],[ (ci_high-uplift)*100 ]],fmt='o',capsize=6)
ax.axhline(0,linestyle='--'); ax.set_xticks([0],['Incremental uplift']); ax.set_ylabel('Percentage points'); ax.set_title('Uplift with 95% confidence interval')
plt.tight_layout(); plt.show()

## 6. Conclusion template
- Did the campaign increase conversion versus control?
- Is the uplift statistically supported, and how wide is its interval?
- How many incremental orders and how much incremental margin were estimated?
- Did margin cover message costs?
- What segment/channel should be tested next?

**Important:** synthetic data and simplified ROI assumptions. In a real campaign, define the attribution window before launch, ensure random assignment and analyze all assigned clients (intent-to-treat).